# 04 — Score agent predictions

Run **`midi-agent score`** on predictions from notebook **03**. **CPU OK.**

In [ ]:
# @title Configuration
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
MIDI_AGENT_REPO = "https://github.com/juliagsy/midi-agent.git"
BRANCH = "main"  # @param {type:"string"}

WORK = "/content"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
MIDI_AGENT = f"{WORK}/midi-agent"

DATASET_MODE = "real_v0.2"  # @param ["pilot", "real_v0.2"]
RUN_ID = "real_v0.2_v1" if DATASET_MODE == "real_v0.2" else "pilot_v1"  # @param {type:"string"}
VARIANT = "g1"  # @param {type:"string"}
AGENT_RUN_SUFFIX = ""  # @param {type:"string"}
SPLIT = "test"  # @param ["test", "validation"]

LOCAL_DATA = f"{MUSICINSTRUCT}/data/pilot" if DATASET_MODE == "pilot" else f"{MUSICINSTRUCT}/data/v0.2"
MANIFEST_NAME = "pilot.jsonl" if DATASET_MODE == "pilot" else "manifest.jsonl"
MANIFEST = f"{LOCAL_DATA}/{MANIFEST_NAME}"
RUN_FOLDER = f"agent_{VARIANT}" + (f"_{AGENT_RUN_SUFFIX}" if AGENT_RUN_SUFFIX else "")

USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_MI = "/content/drive/MyDrive/musicinstruct"
DRIVE_DATA = f"{DRIVE_MI}/data/pilot" if DATASET_MODE == "pilot" else f"{DRIVE_MI}/data/v0.2"
DRIVE_AGENT = "/content/drive/MyDrive/midi-agent"
DRIVE_RESULTS = f"{DRIVE_AGENT}/runs/{RUN_ID}/{RUN_FOLDER}"

In [ ]:
# @title Bootstrap
import shutil
import subprocess
from pathlib import Path

def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

for path, url in [(MUSICINSTRUCT, MUSICINSTRUCT_REPO), (MIDI_AGENT, MIDI_AGENT_REPO)]:
    if Path(path).is_dir():
        shutil.rmtree(path)
    sh(f"git clone --branch {BRANCH} --depth 1 {url} {path}")
sh(f"pip install -q -e {MUSICINSTRUCT}")
sh(f"pip install -q -e {MIDI_AGENT}")

Path(LOCAL_DATA).mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)

pred = Path(DRIVE_RESULTS) / "predictions.jsonl"
if not pred.is_file():
    raise FileNotFoundError(f"Missing {pred} — run notebook 03 first.")
LOCAL_PRED = f"{WORK}/predictions.jsonl"
shutil.copy2(pred, LOCAL_PRED)

In [ ]:
# @title Score
import json
import subprocess
from pathlib import Path

out = Path(f"{WORK}/score_report.json")
subprocess.run(
    ["midi-agent", "score", MANIFEST, LOCAL_PRED, "--split", SPLIT, "--output", str(out)],
    check=True,
)
report = json.loads(out.read_text())
print("overall:", report.get("overall"))
print("composition_macro:", report.get("composition_macro"))

In [ ]:
# @title Sync score report
import shutil
from pathlib import Path

if USE_DRIVE:
    Path(DRIVE_RESULTS).mkdir(parents=True, exist_ok=True)
    shutil.copy2(Path(f"{WORK}/score_report.json"), Path(DRIVE_RESULTS) / "score_report.json")
    print("Saved to", DRIVE_RESULTS)